In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")

print("Libraries imported successfully.")

In [ ]:
df = pd.read_csv('dirty_cafe_sales.csv')
print("Data loaded successfully.")

# Data inspection
df.head(40)

In [ ]:
# Show non-null info
df.info()

In [ ]:
# Cleaning

# Handling missing values

error_values = ["ERROR", "UNKNOWN", "NaN"]

# Handle Numerical columns
numerical_cols = df.select_dtypes(include=[np.number]).columns
for col in numerical_cols:
    median_value = df[col].median()
    df[col].fillna(median_value, inplace=True)
    df[col].replace(error_values, median_value, inplace=True)


# Handle Categorical columns
categorical_cols = df.select_dtypes(include=['object']).columns
for col in categorical_cols:
    mode_value = df[col].mode()[0] if not df[col].mode().empty else 'Unknown'
    df[col].fillna(mode_value, inplace=True)
    df[col].replace(error_values, mode_value, inplace=True)

# Handle Transaction Date column
date_cols = ['Transaction Date']
median_date = pd.to_datetime(df['Transaction Date'], errors='coerce').median()
df["Transaction Date"] = pd.to_datetime(df["Transaction Date"], errors='coerce')
df["Transaction Date"].fillna(median_date, inplace=True)

# Parse the numerical values
# Convert Quantity to int
df['Quantity'] = df['Quantity'].astype(int)

# Convert Total Spent and Price Per Unit to float
df['Total Spent'] = df['Total Spent'].astype(float)
df["Price Per Unit"] = df['Price Per Unit'].astype(float)

print("Missing values handled.")
print("=" * 40)
print("Post-cleaning data info:")
df.info()
print("=" * 40)
print("Post-cleaning data preview:")
df.head(40)

In [ ]:
from calendar import month_name
# Feature Engineering

# get the categories of items sold
unique_food_values = df["Item"].unique()
print("Unique items sold:", unique_food_values)

foods = ["Cake", "Cookie", "Salad", "Sandwich"]
print("Food items:", foods)
print("Drink items:", [item for item in unique_food_values if item not in foods])

df["Category"] = df["Item"].apply(lambda x: "Food" if x in foods else "Drink")

# Get the transaction Month
# if date is ERROR or UNKNOWN, fill with NaT
df['Transaction Month'] = pd.to_datetime(df['Transaction Date'], errors='coerce').dt.month_name()
df['Transaction Month'].fillna("No Date", inplace=True)

# Add a day of week column
df['Day of Week'] = pd.to_datetime(df['Transaction Date'], errors='coerce').dt.day_name()

df.head(40)

In [ ]:
# Analysis

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Total Sold By Item
total_sold_by_item = df.groupby('Item')['Quantity'].sum().reset_index().sort_values(by='Quantity', ascending=False)

axes[0, 0].bar(total_sold_by_item['Item'], total_sold_by_item['Quantity'], color='skyblue')
axes[0, 0].set_title('Total Sold By Item')
axes[0, 0].set_xlabel('Item')
axes[0, 0].set_ylabel('Total Quantity Sold')
axes[0, 0].tick_params(axis='x', rotation=45)

# Total Per Month
total_per_month = df.groupby('Transaction Month')['Total Spent'].sum().reindex(list(month_name)[1:]).reset_index()

axes[0, 1].plot(total_per_month['Transaction Month'], total_per_month['Total Spent'], color='salmon')
axes[0, 1].set_title('Total Spent Per Month')
axes[0, 1].set_xlabel('Month')
axes[0, 1].set_ylabel('Total Spent')
axes[0, 1].tick_params(axis='x', rotation=45)

# Day of the Week Analysis
sales_by_day = df.groupby('Day of Week')['Total Spent'].sum().reindex(['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday'])

axes[1, 0].bar(sales_by_day.index, sales_by_day.values, color='salmon')
axes[1, 0].set_title('Total Spent by Day of the Week')
axes[1, 0].set_xlabel('Day of the Week')
axes[1, 0].set_ylabel('Total Spent')
axes[1, 0].tick_params(axis='x', rotation=45)

# Distribution by Category
category_counts = df['Category'].value_counts()

axes[1, 1].pie(category_counts.values, labels=category_counts.index, autopct='%1.1f%%', startangle=90, colors=['skyblue', 'salmon'])
axes[1, 1].set_title('Distribution by Category')

plt.tight_layout()
plt.show()